In [ ]:
import time
import tracemalloc
from scipy.optimize import curve_fit
import numpy as np

import matplotlib.pyplot as plt

plt.style.use('default')
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'
plt.rcParams['savefig.facecolor'] = 'white'
plt.rcParams['savefig.edgecolor'] = 'white'

plt.rcParams['figure.dpi'] = 500
plt.rcParams['savefig.dpi'] = 500
plt.subplots_adjust(left=0, right=1, top=1, bottom=0)

from pathlib import Path
import importlib.util
import sys
import tempfile
import urllib.request

_reference_util_path = next(
    (
        candidate
        for base in (Path.cwd(), *Path.cwd().parents)
        for candidate in (
            base / "capitulo4" / "referencias" / "util.py",
            base / "referencias" / "util.py",
        )
        if candidate.exists()
    ),
    None,
)
if _reference_util_path is None:
    _reference_util_path = Path(tempfile.gettempdir()) / "capitulo4_reference_util.py"
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/Notas-a-Mano-serie-de-libros/"
        "3_notas-a-mano-sobre-analisis-de-complejidad-computacional/"
        "main/capitulo4/referencias/util.py",
        _reference_util_path,
    )
_reference_spec = importlib.util.spec_from_file_location(
    "capitulo4_reference_util", _reference_util_path
)
_reference_util = importlib.util.module_from_spec(_reference_spec)
sys.modules[_reference_spec.name] = _reference_util
_reference_spec.loader.exec_module(_reference_util)
graficar_complejidad = _reference_util.graficar_complejidad
modelo_constante = _reference_util.modelo_constante
modelo_lineal = _reference_util.modelo_lineal
modelo_cuadratico = _reference_util.modelo_cuadratico


In [ ]:
def medir_tiempo_matrices_param(func, sizes, n_iter):
    tiempos = np.zeros(len(sizes))

    for i, dim in enumerate(sizes):
        acumulado = 0.0
        for _ in range(n_iter):
            inicio = time.perf_counter()
            func(dim, dim)
            acumulado += time.perf_counter() - inicio
        tiempos[i] = acumulado / n_iter

    return tiempos

def medir_memoria_matrices_param(func, sizes, n_iter):
    espacios = np.zeros(len(sizes), dtype=int)

    for i, dim in enumerate(sizes):
        acumulado = 0
        for _ in range(n_iter):
            tracemalloc.start()
            func(dim, dim)
            _, peak = tracemalloc.get_traced_memory()
            tracemalloc.stop()
            acumulado += peak
        espacios[i] = acumulado / n_iter

    return espacios

def imprimir_matriz_creada(m, n):
    matriz = np.random.randint(0, 1000, size=(m, n))
    for fila in matriz:
        for x in fila:
            _ = x

In [ ]:
n_ejecuciones = 250
sizes = np.arange(1, 250, 20)
x = sizes

In [ ]:

tiempos = medir_tiempo_matrices_param(imprimir_matriz_creada, sizes, n_ejecuciones)
params_tiempo = curve_fit(modelo_cuadratico, sizes**2, tiempos)[0]
tiempos_ajustados = modelo_cuadratico(sizes**2, *params_tiempo)

graficar_complejidad(
    x=sizes,
    y_experimental=tiempos,
    y_teorico=tiempos_ajustados,
    nombre_archivo='ejemplo_inicializar_matriz_tiempo.png',
    ylabel='Tiempo de ejecución [s]',
    funcion='T(n)',
    xlabel='Cantidad de datos de entrada ($m \\cdot n$)'
)

In [ ]:

espacio = medir_memoria_matrices_param(imprimir_matriz_creada, sizes, n_ejecuciones)
params_espacio = curve_fit(modelo_cuadratico, sizes, espacio)[0]
espacio_ajustado = modelo_cuadratico(sizes, *params_espacio)

graficar_complejidad(
    x=sizes,
    y_experimental=espacio,
    y_teorico=espacio_ajustado,
    nombre_archivo='ejemplo_inicializar_matriz_espacio.png',
    ylabel='Consumo de memoria [bytes]',
    funcion='S(n)',
    xlabel='Cantidad de datos de entrada ($m \\cdot n$)'
)